# Lab 7: Observability and Evaluation Stack
**Goal:** make a RAG assistant *observable* (you can see what it did and how long it took) and *evaluated* (you can measure whether it is correct, grounded, fast and affordable), then use those numbers as a release gate.

**Why:** an LLM system can fail silently. It does not crash; it gives a fluent wrong answer. Without traces and evals you find out from a customer. This is Module 8, sections 2 (traces, metrics, logs), 3 (evals), 8 (open-source tools such as Langfuse) and 10 (production dashboard).

You will:
1. build a small RAG assistant (a short recap of Lab 3),
2. record a **trace** for every question,
3. run a **golden set** of 20 questions,
4. score each answer with an **LLM judge**,
5. summarise latency, estimated cost and quality, and apply pass/fail **gates**,
6. optionally send the same traces to **Langfuse**.

## Exercise 0: Choose your model provider
**Why we do this:** the same provider setup as Labs 4 and 6. This lab makes about 40 model calls per run (20 answers and 20 judgements), so a free tier with tight limits may slow you down. You can lower `LIMIT` below to run fewer questions.

**What the code does:** defines `chat(messages)`, then an `ask(system, user)` helper with a few retries and a short wait, because rate-limit errors are common when you run many calls in a row.

Set `PROVIDER` first. Ollama must run locally (Colab cannot reach it). Model names change, so edit `MODEL` if one is rejected.

In [ ]:
%pip install -q chromadb pypdf openai anthropic

In [ ]:
import os, json, getpass

PROVIDER = "groq"   # ollama | groq | gemini | openrouter | openai | anthropic

PROVIDERS = {
    "ollama":     {"base_url": "http://localhost:11434/v1",                                "key_env": None,                "model": "llama3.1:8b"},
    "groq":       {"base_url": "https://api.groq.com/openai/v1",                          "key_env": "GROQ_API_KEY",       "model": "llama-3.3-70b-versatile"},
    "gemini":     {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", "key_env": "GEMINI_API_KEY",     "model": "gemini-2.5-flash"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",                            "key_env": "OPENROUTER_API_KEY", "model": "openrouter/free"},
    "openai":     {"base_url": None,                                                      "key_env": "OPENAI_API_KEY",     "model": "gpt-4o-mini"},
    "anthropic":  {"base_url": None,                                                      "key_env": "ANTHROPIC_API_KEY",  "model": "claude-sonnet-5-5"},
}
cfg = PROVIDERS[PROVIDER]
MODEL = cfg["model"]
if cfg["key_env"] and not os.getenv(cfg["key_env"]):
    os.environ[cfg["key_env"]] = getpass.getpass(cfg["key_env"] + ": ")

def to_anthropic(messages):
    # Convert OpenAI-style messages into Anthropic's format.
    system = messages[0]["content"]
    out = []
    for m in messages[1:]:
        if m["role"] == "user":
            out.append({"role": "user", "content": m["content"]})
        elif m["role"] == "assistant":
            blocks = []
            if m.get("content"):
                blocks.append({"type": "text", "text": m["content"]})
            for tc in m.get("tool_calls", []):
                blocks.append({"type": "tool_use", "id": tc["id"], "name": tc["function"]["name"],
                               "input": json.loads(tc["function"]["arguments"] or "{}")})
            out.append({"role": "assistant", "content": blocks})
        elif m["role"] == "tool":
            block = {"type": "tool_result", "tool_use_id": m["tool_call_id"], "content": m["content"]}
            if out and out[-1]["role"] == "user" and isinstance(out[-1]["content"], list):
                out[-1]["content"].append(block)
            else:
                out.append({"role": "user", "content": [block]})
    return system, out

def chat(messages, tools=None):
    # Returns {"role": "assistant", "content": str, "tool_calls": [...] (optional)}
    if PROVIDER == "anthropic":
        import anthropic
        system, msgs = to_anthropic(messages)
        kw = dict(model=MODEL, max_tokens=1000, temperature=0, system=system, messages=msgs)
        if tools:
            kw["tools"] = [{"name": t["function"]["name"], "description": t["function"]["description"],
                            "input_schema": t["function"]["parameters"]} for t in tools]
        r = anthropic.Anthropic().messages.create(**kw)
        msg = {"role": "assistant", "content": "".join(b.text for b in r.content if b.type == "text")}
        calls = [{"id": b.id, "type": "function", "function": {"name": b.name, "arguments": json.dumps(b.input)}}
                 for b in r.content if b.type == "tool_use"]
        if calls:
            msg["tool_calls"] = calls
        return msg
    from openai import OpenAI
    key = os.getenv(cfg["key_env"]) if cfg["key_env"] else "ollama"
    client = OpenAI(base_url=cfg["base_url"], api_key=key) if cfg["base_url"] else OpenAI(api_key=key)
    kw = dict(model=MODEL, temperature=0, messages=messages)
    if tools:
        kw["tools"] = tools
    m = client.chat.completions.create(**kw).choices[0].message
    msg = {"role": "assistant", "content": m.content or ""}
    if m.tool_calls:
        msg["tool_calls"] = [{"id": tc.id, "type": "function",
                              "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                             for tc in m.tool_calls]
    return msg

print(chat([{"role": "system", "content": "Be brief."}, {"role": "user", "content": "Say hello in five words."}])["content"])

In [ ]:
import time

def ask(system: str, user: str, retries: int = 3) -> str:
    for attempt in range(retries):
        try:
            return chat([{"role": "system", "content": system}, {"role": "user", "content": user}])["content"]
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** (attempt + 1))   # back off: 2s, 4s

LIMIT = 20    # how many golden questions to run (lower it to save calls)
PAUSE = 0.5   # seconds to wait between questions (raise it on strict free tiers)

## Exercise 1: Build the assistant (recap of Lab 3)
**Why we do this:** you need something to observe. This is a compact version of Lab 3: download the SEBI memo, chunk it, store it in ChromaDB, retrieve, and answer from the retrieved text only.

**What the code does:** `retrieve_chunks` returns the top matching chunks with page numbers. `generate` builds a prompt from them and asks the model, replying "Not found in the provided documents." when the context does not contain the answer. It returns both the answer and the prompt, because the next exercise needs the prompt to estimate tokens.

**Key terms:** *Token* = the unit models read and are billed in, roughly three-quarters of an English word.

In [ ]:
import re, urllib.request
import chromadb
from pypdf import PdfReader

os.makedirs("data", exist_ok=True)
SEBI_URL = "https://www.sebi.gov.in/sebi_data/meetingfiles/dec-2024/1735042007618_1.pdf"
if not os.path.exists("data/sebi_ai_memo.pdf"):
    req = urllib.request.Request(SEBI_URL, headers={"User-Agent": "Mozilla/5.0"})
    open("data/sebi_ai_memo.pdf", "wb").write(urllib.request.urlopen(req).read())

PAGES = {i + 1: re.sub(r"\s+", " ", p.extract_text() or "").strip() for i, p in enumerate(PdfReader("data/sebi_ai_memo.pdf").pages)}

def chunk(text, size=600, overlap=100):
    return [text[s:s + size] for s in range(0, max(len(text), 1), size - overlap) if text[s:s + size].strip()]

col = chromadb.Client().get_or_create_collection("lab7")
ids, docs, metas = [], [], []
for pg, text in PAGES.items():
    for j, piece in enumerate(chunk(text)):
        ids.append(f"p{pg}-{j}"); docs.append(piece); metas.append({"page": pg})
col.add(ids=ids, documents=docs, metadatas=metas)

def retrieve_chunks(question, k=4):
    r = col.query(query_texts=[question], n_results=k)
    return [{"text": t, "page": m["page"]} for t, m in zip(r["documents"][0], r["metadatas"][0])]

SYSTEM = ("You answer questions about the SEBI memo using ONLY the numbered sources. Cite pages like (p.3). "
          "If the sources do not contain the answer, reply exactly: 'Not found in the provided documents.'")

def generate(question, chunks):
    sources = "\n\n".join(f"[{i+1}] (p.{c['page']}) {c['text']}" for i, c in enumerate(chunks))
    prompt = f"Sources:\n{sources}\n\nQuestion: {question}"
    return {"answer": ask(SYSTEM, prompt), "prompt": SYSTEM + "\n" + prompt}

q = "In which year did SEBI circulars mandate reporting of AI and ML applications?"
g = generate(q, retrieve_chunks(q))
print(g["answer"])

**Expected output:** an answer mentioning 2019, with a page citation. If it says "Not found", check that the chunk count is above zero and try a stronger model.

## Exercise 2: Add tracing
**Why we do this:** when an answer is wrong or slow you need to know *where*: did retrieval fail, or did generation? A **trace** records one request from start to finish. Inside it, each step is a **span** with its own timing and details (Module 8, section 2).

**What the code does:** the `Trace` class below records spans. Its `span(...)` context manager times whatever runs inside the `with` block and stores the result. `finish` adds up time, estimated tokens and estimated cost for the whole request.

**Key terms:**
- *Trace:* the full record of one request.
- *Span:* one timed step inside a trace.
- *Estimated tokens:* we use about 4 characters per token. These are **estimates, not billed numbers**. Your provider's usage fields are exact (a stretch goal).

**Your task:** complete `rag_answer` so it records a `retrieve` span and a `generate` span. In the generate span, store `in_tok` and `out_tok` in the span dictionary.

**Cost:** set `PRICE_IN_PER_M` and `PRICE_OUT_PER_M` (dollars per million tokens) from your provider's current pricing page. Leave them at 0 for free tiers.

In [ ]:
import uuid, contextlib, math

PRICE_IN_PER_M = 0.0    # dollars per million input tokens (set from your provider's pricing page)
PRICE_OUT_PER_M = 0.0   # dollars per million output tokens

def est_tokens(text: str) -> int:
    return max(1, len(text) // 4)

TRACES = []

class Trace:
    def __init__(self, name, question):
        self.id = uuid.uuid4().hex[:8]
        self.name, self.question = name, question
        self.spans, self.output = [], None
        self.total_ms = self.in_tok = self.out_tok = 0
        self.cost = 0.0

    @contextlib.contextmanager
    def span(self, name, **attrs):
        s = {"name": name, **attrs}
        t0 = time.time()
        try:
            yield s
        finally:
            s["ms"] = round((time.time() - t0) * 1000)
            self.spans.append(s)

    def finish(self, output):
        self.output = output
        self.total_ms = sum(s["ms"] for s in self.spans)
        self.in_tok = sum(s.get("in_tok", 0) for s in self.spans)
        self.out_tok = sum(s.get("out_tok", 0) for s in self.spans)
        self.cost = (self.in_tok * PRICE_IN_PER_M + self.out_tok * PRICE_OUT_PER_M) / 1_000_000
        TRACES.append(self)

def rag_answer(question, k=4):
    trace = Trace("rag_answer", question)
    # YOUR CODE HERE:
    # 1. with trace.span("retrieve", k=k) as s: retrieve the chunks, then set s["n_chunks"]
    # 2. with trace.span("generate", model=MODEL) as s: call generate(question, chunks),
    #    then set s["in_tok"] and s["out_tok"] using est_tokens on the prompt and the answer
    # 3. call trace.finish(answer) and return {"answer": ..., "chunks": ..., "trace": trace}
    raise NotImplementedError

<details><summary>Solution (try first)</summary>

```python
def rag_answer(question, k=4):
    trace = Trace("rag_answer", question)
    with trace.span("retrieve", k=k) as s:
        chunks = retrieve_chunks(question, k)
        s["n_chunks"] = len(chunks)
    with trace.span("generate", model=MODEL) as s:
        g = generate(question, chunks)
        s["in_tok"] = est_tokens(g["prompt"])
        s["out_tok"] = est_tokens(g["answer"])
    trace.finish(g["answer"])
    return {"answer": g["answer"], "chunks": chunks, "trace": trace}
```
</details>

In [ ]:
r = rag_answer("Which circular covered mutual funds and what was its date?")
t = r["trace"]
print(r["answer"], "\n")
print("trace", t.id, "| total", t.total_ms, "ms | est tokens in/out:", t.in_tok, "/", t.out_tok)
for s in t.spans:
    print("  span:", s)

**Expected output:** the answer, then one line per span showing its name, timing in milliseconds, and details. The `generate` span will almost always be far slower than `retrieve`.

**What to notice:** that gap tells you where optimisation effort would pay off. If `retrieve` were the slow one, you would tune the database, not the model.

**Finance link:** a trace is also an audit record: which question, which sources, which model, when (Module 8, section 9).

## Exercise 3: The golden set
**Why we do this:** you cannot measure quality without a fixed set of questions with known right answers. This is a *golden set*. Running the same set after every change shows whether you made things better or worse (a *regression test*).

**What the code does:** 20 questions written from the SEBI memo you loaded. 18 have answers in the document. 2 do not (the memo excises the number of comments received, and states no penalty). For those, the *correct* behaviour is to say "Not found". Testing that is important: a system that always answers is dangerous in finance.

Each item has `keys`: a list of groups. An answer passes if, for every group, at least one alternative appears in it (case-insensitive). This is a simple, repeatable fact check.

**Your task:** add two questions of your own at the end of the list (with `keys`), taken from the memo. Check the memo text first so the expected answer is right.

**Important:** a golden set of 20 is enough to teach the idea and to catch big regressions, but it is far too small to detect small quality changes. Real teams grow it from production failures.

In [ ]:
GOLDEN = [
    {"q": "Which three SEBI regulations does the memo propose to amend?", "answerable": True,
     "keys": [["intermediaries"], ["stock exchanges"], ["depositories"]]},
    {"q": "What is the date of the SEBI consultation paper on this proposal?", "answerable": True,
     "keys": [["november 13, 2024", "13 november 2024", "13th november 2024", "nov 13, 2024", "13/11/2024"]]},
    {"q": "In which year did SEBI circulars first mandate reporting of AI and ML applications?", "answerable": True,
     "keys": [["2019"]]},
    {"q": "What is the date of the circular covering stock brokers and depository participants?", "answerable": True,
     "keys": [["january 4, 2019", "4 january 2019", "4th january 2019"]]},
    {"q": "What is the date of the circular covering market infrastructure institutions?", "answerable": True,
     "keys": [["january 31, 2019", "31 january 2019", "january, 31, 2019"]]},
    {"q": "What is the date of the circular covering mutual funds and asset management companies?", "answerable": True,
     "keys": [["may 9, 2019", "9 may 2019"]]},
    {"q": "For which purposes do stock brokers and depository participants use AI/ML tools?", "answerable": True,
     "keys": [["surveillance"], ["order execution"], ["kyc"]]},
    {"q": "For which purposes do mutual funds use AI/ML tools?", "answerable": True,
     "keys": [["surveillance"], ["customer service"], ["customer segmentation"]]},
    {"q": "Which kinds of institutions are market infrastructure institutions?", "answerable": True,
     "keys": [["stock exchange"], ["depositor"], ["clearing corporation"]]},
    {"q": "What three things would a regulated person be solely responsible for under the proposal?", "answerable": True,
     "keys": [["privacy", "data"], ["output"], ["compliance", "applicable laws"]]},
    {"q": "Does the proposed responsibility depend on the scale of AI adoption?", "answerable": True,
     "keys": [["irrespective", "regardless", "does not depend", "not depend", "any scale", "scale and scenario"]]},
    {"q": "Does the proposal cover AI tools bought from third-party providers?", "answerable": True,
     "keys": [["third-party", "third party"]]},
    {"q": "What does the memo ask the Board to approve?", "answerable": True,
     "keys": [["amend"], ["chairperson"]]},
    {"q": "Why was the AI/ML reporting mechanism introduced?", "answerable": True,
     "keys": [["adoption"], ["preparedness", "prepared"]]},
    {"q": "Name some benefits of AI that the memo lists.", "answerable": True,
     "keys": [["efficiency"], ["cyber security", "cybersecurity"], ["customer experience"]]},
    {"q": "Which AI use by market infrastructure institutions involves generative AI?", "answerable": True,
     "keys": [["generative"], ["document", "report", "order"]]},
    {"q": "Where is the consultation paper available?", "answerable": True,
     "keys": [["annexure a", "reports for public comments", "website", "sebi.gov.in"]]},
    {"q": "Why does SEBI want to assign responsibility to regulated persons using AI?", "answerable": True,
     "keys": [["investor"], ["serious"]]},
    {"q": "What penalty will apply to a regulated person who breaches the proposed rule?", "answerable": False, "keys": []},
    {"q": "How many public comments were received on the consultation paper?", "answerable": False, "keys": []},
    # YOUR CODE HERE: add two questions of your own from the memo, in the same format
]

def check_facts(answer: str, keys) -> bool:
    a = answer.lower()
    return all(any(alt in a for alt in group) for group in keys)

print(len(GOLDEN), "golden questions,", sum(g["answerable"] for g in GOLDEN), "answerable")

**Expected output:** `20 golden questions, 18 answerable` (more if you added your own).

**What to notice:** the fact check is strict about wording. A correct answer phrased in an unexpected way would fail it. That is why we also use an LLM judge, and why some failures deserve a human look.

## Exercise 4: Evaluate with an LLM judge
**Why we do this:** the fact check tells you if the key facts are present. It does not tell you if the answer *invented* anything else. **Groundedness** asks: is every claim in the answer supported by the retrieved text? A model can be correct yet partly invented, and in finance that matters (Module 8, section 3).

**What the code does:** an *LLM judge* is a second model call that reads the sources and the answer and gives a 1 (fully supported) or 0 (some unsupported claim). `evaluate` runs every golden question through `rag_answer`, then records correctness, groundedness, latency and cost.

**Key terms:**
- *LLM-as-judge:* using a model to grade another model's output.
- *Groundedness (faithfulness):* the answer sticks to the supplied sources.

**Your task:** complete `groundedness(context, answer)`. Write a judge prompt that ends with `SCORE: 1` or `SCORE: 0` followed by `REASON:`, call `ask`, and parse the score with a regular expression. Return 1 or 0, and return 0 if no score can be parsed.

**Important:** judges are imperfect. They can be lenient with fluent text, and if the judge is the same model as the generator it tends to favour its own style. Use a different, stronger model as judge when you can. Spot-check a few judgements by hand.

In [ ]:
def groundedness(context: str, answer: str) -> int:
    # YOUR CODE HERE:
    # 1. build a system prompt telling the judge to check ONLY whether every factual claim
    #    in the answer is supported by the context (an answer saying 'Not found' is grounded)
    # 2. ask it to reply with 'SCORE: 1' or 'SCORE: 0' then 'REASON: <one sentence>'
    # 3. parse with re.search(r"SCORE:\s*([01])", reply); return int(...) or 0 if no match
    raise NotImplementedError

def percentile(values, p):
    s = sorted(values)
    return s[max(0, math.ceil(p / 100 * len(s)) - 1)]

def evaluate(items, k=4, label="run"):
    rows = []
    for it in items[:LIMIT]:
        r = rag_answer(it["q"], k)
        ans, t = r["answer"], r["trace"]
        correct = check_facts(ans, it["keys"]) if it["answerable"] else ("not found" in ans.lower())
        context = "\n".join(ch["text"] for ch in r["chunks"])
        rows.append({"q": it["q"], "answer": ans, "correct": int(correct),
                     "grounded": groundedness(context, ans), "ms": t.total_ms, "cost": t.cost})
        time.sleep(PAUSE)
    print(f"[{label}] evaluated {len(rows)} questions")
    return rows

<details><summary>Solution (try first)</summary>

```python
def groundedness(context, answer):
    system = ("You are a strict fact-checker. Decide ONLY whether every factual claim in the ANSWER "
              "is supported by the CONTEXT. An answer saying 'Not found in the provided documents.' is grounded. "
              "Reply with 'SCORE: 1' if all claims are supported or 'SCORE: 0' if any is not, "
              "then 'REASON:' and one sentence.")
    reply = ask(system, f"CONTEXT:\n{context}\n\nANSWER:\n{answer}")
    m = re.search(r"SCORE:\s*([01])", reply)
    return int(m.group(1)) if m else 0
```
</details>

In [ ]:
rows = evaluate(GOLDEN, k=4, label="baseline k=4")
for r in rows:
    flag = "ok " if r["correct"] and r["grounded"] else "FAIL"
    print(f"{flag}  correct={r['correct']} grounded={r['grounded']} {r['ms']:>5}ms  {r['q'][:70]}")

**Expected output:** one line per question. Most should pass; a few `FAIL` lines are normal and are the point.

**What to notice:** read two or three failing answers in full (`rows[i]["answer"]`). Decide for yourself: is it a real error, a retrieval miss, a too-strict fact check, or a judge mistake? Those four causes call for four different fixes.

## Exercise 5: Summary, gates and a regression test
**Why we do this:** 20 rows are hard to act on. A production dashboard reduces them to a few numbers with thresholds, so a person (or a CI job) can say "ship" or "do not ship" (Module 8, section 10).

**What the code does:** you write `summarise(rows)` to compute correctness rate, groundedness rate, median and 95th-percentile latency, and cost per query. `gates` then compares them with thresholds. **These thresholds are illustrative.** Set yours from your own risk appetite and review them with your risk team.

**Key terms:**
- *p50 / p95 latency:* half / 95% of requests finish faster than this. p95 shows the slow tail that users actually notice.
- *Gate:* a pass/fail threshold that blocks a release.
- *Regression test:* re-running the golden set after a change to see if anything got worse.

**Your task:** complete `summarise`. Hint: use `sum(...) / len(rows)` for the rates, and the `percentile` helper for latency.

In [ ]:
def summarise(rows):
    # YOUR CODE HERE: return a dict with keys
    # correctness, groundedness, p50_ms, p95_ms, cost_per_query
    raise NotImplementedError

THRESHOLDS = {"correctness": 0.80, "groundedness": 0.90, "p95_ms": 8000, "cost_per_query": 0.01}   # illustrative

def gates(s):
    checks = {
        "correctness >= 0.80":     s["correctness"] >= THRESHOLDS["correctness"],
        "groundedness >= 0.90":    s["groundedness"] >= THRESHOLDS["groundedness"],
        "p95 latency <= 8000 ms":  s["p95_ms"] <= THRESHOLDS["p95_ms"],
        "cost/query <= $0.01":     s["cost_per_query"] <= THRESHOLDS["cost_per_query"],
    }
    for name, ok in checks.items():
        print(("PASS  " if ok else "FAIL  ") + name)
    print("RELEASE:", "OK" if all(checks.values()) else "BLOCKED")

<details><summary>Solution (try first)</summary>

```python
def summarise(rows):
    n = len(rows)
    return {
        "correctness": sum(r["correct"] for r in rows) / n,
        "groundedness": sum(r["grounded"] for r in rows) / n,
        "p50_ms": percentile([r["ms"] for r in rows], 50),
        "p95_ms": percentile([r["ms"] for r in rows], 95),
        "cost_per_query": sum(r["cost"] for r in rows) / n,
    }
```
</details>

In [ ]:
base = summarise(rows)
print({k: round(v, 4) for k, v in base.items()})
gates(base)

**Now run a regression test.** Change one setting, here retrieving only 1 chunk instead of 4, and compare. This simulates a "cost-saving" change that someone might ship without testing.

In [ ]:
rows_k1 = evaluate(GOLDEN, k=1, label="candidate k=1")
cand = summarise(rows_k1)
print(f"{'metric':16}{'baseline':>12}{'candidate':>12}")
for k in base:
    print(f"{k:16}{base[k]:>12.3f}{cand[k]:>12.3f}")
gates(cand)

**Expected output:** a side-by-side table. With only one chunk, correctness usually drops (questions needing several pieces of the memo suffer), while latency and cost may improve slightly.

**What to notice:** without the golden set you would see only the lower cost and think the change was free. With only 20 questions, treat small differences as noise and big drops as real.

**Finance link:** this before-and-after table is the kind of validation evidence that model-risk reviewers expect for any change to a model-driven process (Module 5, section 2; Module 8, section 9).

## Exercise 6 (optional): Send traces to Langfuse
**Why we do this:** your `Trace` class teaches what a trace is. In practice you use a tool that stores traces, draws them as a timeline, and lets you search and compare them. **Langfuse** is an open-source option (Module 8, section 8).

**Two ways to run it:**
- **Langfuse Cloud (works on Colab).** Create a free account and project at langfuse.com, then copy the public key, secret key and host URL. **Warning:** traces include your questions and answers and are sent to a third party. Never send real client data. Use only fake or public data like this lab's.
- **Self-hosted with Docker (keeps data on your machine).** Docker must be installed. Clone the Langfuse repository and start it with `docker compose up`, following the current self-hosting guide on the Langfuse site. It runs several services and needs a few GB of memory, and it cannot be reached from Colab, so use a local notebook. The UI is normally at http://localhost:3000, where you create a project and keys.

**What the code does:** reads your keys, then wraps the existing `retrieve_chunks` and `generate` functions with Langfuse's `observe` decorator. Because `rag_answer` calls them, each question now appears as a trace with nested `retrieve` and `generate` steps.

**Important:** the Langfuse SDK changes between versions. This cell was written for the v3 Python SDK (`from langfuse import observe, get_client`). If an import or call fails, check the current Langfuse documentation. The rest of the lab does not depend on this cell.

In [ ]:
%pip install -q langfuse

In [ ]:
LANGFUSE_PUBLIC_KEY = ""   # paste from your Langfuse project (leave empty to skip this exercise)
LANGFUSE_SECRET_KEY = ""
LANGFUSE_HOST = "https://cloud.langfuse.com"   # or your region's URL, or http://localhost:3000 when self-hosting

if LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY:
    os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
    os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
    os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST
    try:
        from langfuse import observe, get_client
        retrieve_chunks = observe(name="retrieve")(retrieve_chunks)
        generate = observe(name="generate")(generate)
        for it in GOLDEN[:3]:
            rag_answer(it["q"])
        get_client().flush()
        print("Sent 3 traces. Open your Langfuse project and look at the Traces page.")
    except Exception as e:
        print("Langfuse step skipped:", type(e).__name__, e)
else:
    print("No Langfuse keys set. Skipping this optional exercise.")

**Expected output:** "Sent 3 traces", and the Traces page in Langfuse showing three entries, each with its timing.

**What to notice:** compare Langfuse's timeline with the spans your own `Trace` recorded. Your class already stores the same facts. The tool adds storage, search, dashboards and team sharing.

**Finance link:** question text often contains personal data. Before sending traces anywhere, redact them (reuse `redact` from Lab 6) and check your retention and residency rules (Module 8, section 9).

## Stretch goals
1. Use your provider's exact token counts instead of the 4-characters-per-token estimate, and compare the two.
2. Use a different, stronger model as the judge and see how many judgements change.
3. Try DeepEval or RAGAS, open-source evaluation libraries, to compute faithfulness and answer relevance, and compare with your own judge.
4. Redact personal data from traces before logging them.
5. Add a latency or groundedness check that fails a script with a non-zero exit code, so a CI pipeline can block a release.

## Common errors
- **`NotImplementedError`:** you have not completed `rag_answer`, `groundedness` or `summarise` yet. Complete them in order.
- **Rate limit or 429 errors:** raise `PAUSE`, lower `LIMIT`, or use a provider with higher limits. Each question uses two model calls.
- **Groundedness is 0 for everything:** your judge reply is probably not in the `SCORE: 1` format. Print one raw reply and adjust the prompt or the regular expression.
- **Langfuse import errors:** the SDK API changes between versions. Check the current docs, or skip Exercise 6.
- **Scores change between runs:** expected. Model output varies, so repeat the run before drawing conclusions.